# Offline Bangla HTR Model Training & Interactive Monitoring Notebook

This Jupyter notebook allows you to run and visually monitor the **3-Stage HTR Training Pipeline** with live progress bars, real-time loss plots, and CER/WER metrics directly inside VS Code / Jupyter.

In [ ]:
import os
import time
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Enable MPS Fallback for Apple Silicon
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from dataset import BanglaTokenizer, LocalBanglaDataset, SyntheticBanglaDataset, HTRCollateFn
from model import HTRHybridModel
from dictionary import BanglaPostProcessor
from rl_agent import HTRRewardEngine, RLActorCriticAgent
from train import train_ctc_epoch, evaluate_model, get_device

device = get_device()
print(f"Active PyTorch Device: {device}")

## Step 1: Initialize Tokenizer & Model Architecture

In [ ]:
metadata_csv = "Bangla dataset/metaData_img.csv"
local_dataset_dir = "Bangla dataset/dataset_filtered"

tokenizer = BanglaTokenizer(metadata_csv)
post_processor = BanglaPostProcessor()
num_classes = len(tokenizer)

print(f"Vocabulary Size: {num_classes} classes")

model = HTRHybridModel(num_classes=num_classes, hidden_dim=256).to(device)
print("Model Architecture Initialized Successfully!")

## Step 2: Load Datasets & Preview Samples

In [ ]:
local_dataset = LocalBanglaDataset(local_dataset_dir, metadata_csv, tokenizer, is_training=True)
print(f"Total Local Handwriting Samples Loaded: {len(local_dataset)}")

# Preview first sample
sample_img, sample_tokens, label_str = local_dataset[0]
plt.figure(figsize=(4, 2))
plt.imshow(sample_img.squeeze(0).cpu().numpy(), cmap="gray")
plt.title(f"Label: {label_str}")
plt.axis("off")
plt.show()

## Step 3: Run Full Training Pipeline with Live Progress

In [ ]:
# Set Hyperparameters
batch_size = 16
pretrain_epochs = 5
finetune_epochs = 10
rl_epochs = 5

local_loader = DataLoader(local_dataset, batch_size=batch_size, shuffle=True, collate_fn=HTRCollateFn)
ctc_loss_fn = torch.nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=finetune_epochs)

history_loss = []
history_cer = []

print("Starting Fine-Tuning Stage...")
for epoch in range(1, finetune_epochs + 1):
    loss = train_ctc_epoch(model, local_loader, optimizer, scheduler, ctc_loss_fn, device, epoch, finetune_epochs, stage_name="Fine-Tune")
    cer, wer, dict_match = evaluate_model(model, local_loader, tokenizer, post_processor, device)
    history_loss.append(loss)
    history_cer.append(cer)
    print(f"Epoch [{epoch}/{finetune_epochs}] | Loss: {loss:.4f} | CER: {cer:.4f} | WER: {wer:.4f} | Lexicon Match: {dict_match*100:.1f}%")

# Plot Training Progress
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_loss, label="CTC Loss", color="blue")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_cer, label="CER", color="red")
plt.xlabel("Epoch")
plt.ylabel("Character Error Rate")
plt.title("CER Progress Curve")
plt.legend()

plt.tight_layout()
plt.show()